# Deep Learning with PyTorch: Interview-Ready Implementations

This notebook covers PyTorch deep learning from scratch, focused on concepts that appear in ML engineering and data science interviews. Each section builds intuition first, then shows code proof.

## Sections
1. PyTorch Fundamentals (the parts that trip up interviewees)
2. Neural Network from Scratch - MLP on MNIST
3. Convolutional Neural Networks
4. Training Tricks
5. 15 Deep Learning Interview Q&A with Code Proof

**Note:** All code uses try/except for torch imports. If PyTorch is not installed, sklearn/numpy equivalents are used automatically.

In [ ]:
# Setup: detect available libraries
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

TORCH_AVAILABLE = False
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    TORCH_AVAILABLE = True
    print(f'PyTorch version: {torch.__version__}')
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {DEVICE}')
except ImportError:
    print('PyTorch not available. Using numpy/sklearn fallbacks.')
    DEVICE = 'cpu'

TORCHVISION_AVAILABLE = False
if TORCH_AVAILABLE:
    try:
        import torchvision
        import torchvision.transforms as transforms
        import torchvision.datasets as datasets
        TORCHVISION_AVAILABLE = True
        print(f'torchvision version: {torchvision.__version__}')
    except ImportError:
        print('torchvision not available. Using sklearn digits dataset.')

from sklearn import datasets as sk_datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
print('sklearn available: OK')
print(f'numpy version: {np.__version__}')

---
## SECTION 1: PyTorch Fundamentals

### 1.1 Tensors vs NumPy Arrays

The key differences interviewers test:
- Tensors live on GPU or CPU; arrays only on CPU
- Tensors track gradients for autograd; arrays do not
- Tensor operations are lazy-differentiable; array operations are not
- Shared memory between numpy and torch (`.numpy()` / `torch.from_numpy()`)

In [ ]:
if TORCH_AVAILABLE:
    # --- Creation ---
    np_arr = np.array([[1.0, 2.0], [3.0, 4.0]])
    t_from_np = torch.from_numpy(np_arr)        # shares memory!
    t_direct  = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # copies data

    print('NumPy array:')
    print(np_arr)
    print('\nTensor from numpy (shared memory):')
    print(t_from_np)
    print('\nDirect tensor:')
    print(t_direct)

    # Modify numpy -> tensor reflects it (shared memory)
    np_arr[0, 0] = 99.0
    print('\nAfter modifying np_arr[0,0] to 99:')
    print('np_arr:', np_arr[0,0])
    print('t_from_np[0,0]:', t_from_np[0,0].item(), '<-- shared memory, also changed!')
    print('t_direct[0,0]:', t_direct[0,0].item(), '<-- copy, unchanged')
else:
    print('(torch not available - showing numpy equivalent)')
    a = np.array([[1.0, 2.0], [3.0, 4.0]])
    print(a)

In [ ]:
if TORCH_AVAILABLE:
    # --- Shape and dtype operations ---
    x = torch.randn(3, 4)   # shape (3,4), dtype float32
    print('Shape:', x.shape)
    print('dtype:', x.dtype)
    print('device:', x.device)

    # Reshape
    x_reshaped = x.view(12)       # flat
    x_reshaped2 = x.reshape(2, 6) # reshape (safer than view)
    print('\nOriginal:', x.shape, '-> view(12):', x_reshaped.shape, '-> reshape(2,6):', x_reshaped2.shape)

    # Type casting
    x_int = x.to(torch.int32)
    x_double = x.double()
    print('\nFloat32 -> int32:', x_int.dtype)
    print('Float32 -> float64:', x_double.dtype)

    # Broadcasting (same rules as numpy)
    a = torch.ones(3, 1)
    b = torch.ones(1, 4)
    print('\nBroadcast (3,1) + (1,4) ->', (a + b).shape)  # (3,4)
else:
    x = np.random.randn(3, 4)
    print('NumPy shape:', x.shape, 'dtype:', x.dtype)

### 1.2 Autograd: How Backpropagation Works

PyTorch builds a **dynamic computational graph** as you do forward operations. When you call `.backward()`, it traverses this graph in reverse (chain rule) to compute gradients.

Key rule: only `float` tensors with `requires_grad=True` participate.

In [ ]:
if TORCH_AVAILABLE:
    # Simple example: y = x^2 + 3x + 2, dy/dx = 2x + 3
    x = torch.tensor(4.0, requires_grad=True)
    y = x**2 + 3*x + 2

    print(f'x = {x.item()}')
    print(f'y = x^2 + 3x + 2 = {y.item()}')
    print(f'Expected dy/dx at x=4: 2*4 + 3 = {2*4+3}')

    y.backward()  # compute gradients
    print(f'Computed dy/dx (x.grad): {x.grad.item()}')

    # Multi-variable: z = x^2 + xy + y^2
    # dz/dx = 2x + y, dz/dy = x + 2y
    x = torch.tensor(2.0, requires_grad=True)
    y_var = torch.tensor(3.0, requires_grad=True)
    z = x**2 + x*y_var + y_var**2

    z.backward()
    print(f'\nz = x^2 + xy + y^2, x=2, y=3')
    print(f'dz/dx = 2x + y = {2*2+3} | computed: {x.grad.item()}')
    print(f'dz/dy = x + 2y = {2+2*3} | computed: {y_var.grad.item()}')
else:
    # Manual gradient with numpy
    x_val = 4.0
    y_val = x_val**2 + 3*x_val + 2
    grad = 2*x_val + 3
    print(f'y = x^2 + 3x + 2 at x=4: {y_val}')
    print(f'dy/dx = 2x + 3 = {grad}')

### 1.3 Computational Graph Visualization (Text-Based)

PyTorch builds a DAG (directed acyclic graph). Each node is an operation; leaves are inputs.

In [ ]:
if TORCH_AVAILABLE:
    x = torch.tensor(2.0, requires_grad=True)
    w = torch.tensor(3.0, requires_grad=True)
    b = torch.tensor(1.0, requires_grad=True)

    # forward: z = w*x + b, loss = z^2
    z = w * x + b
    loss = z ** 2

    print('Computational Graph (forward pass):')
    print()
    print('  x=2.0         w=3.0         b=1.0')
    print('   |              |              |')
    print('   +----[MulBackward0]---+        |')
    print('           |             |        |')
    print('           |        [AddBackward0]+---+')
    print('                         |')
    print('                        z=7.0')
    print('                         |')
    print('                   [PowBackward0]')
    print('                         |')
    print('                      loss=49.0')
    print()
    print(f'z = w*x + b = {z.item()}')
    print(f'loss = z^2 = {loss.item()}')

    loss.backward()
    # dloss/dz = 2z = 14
    # dloss/dw = dloss/dz * dz/dw = 14 * 2 = 28
    # dloss/dx = dloss/dz * dz/dx = 14 * 3 = 42
    # dloss/db = dloss/dz * dz/db = 14 * 1 = 14
    print(f'dloss/dx = {x.grad.item()} (expected: 2*z*w = 2*7*3 = {2*7*3})')
    print(f'dloss/dw = {w.grad.item()} (expected: 2*z*x = 2*7*2 = {2*7*2})')
    print(f'dloss/db = {b.grad.item()} (expected: 2*z*1 = 2*7*1 = {2*7*1})')
else:
    print('Computational graph example (numpy):')
    x, w, b = 2.0, 3.0, 1.0
    z = w * x + b
    loss = z ** 2
    dloss_dz = 2 * z
    print(f'z={z}, loss={loss}')
    print(f'dloss/dx={dloss_dz * w}, dloss/dw={dloss_dz * x}, dloss/db={dloss_dz}')

### 1.4 `.detach()`, `.requires_grad`, `.backward()` — When to Use Each

| Method | When to use |
|--------|-------------|
| `.detach()` | Stop gradient flow (e.g., when using a tensor as target/label, or freezing part of network) |
| `.requires_grad_(False)` | Freeze parameters permanently (e.g., pretrained layers) |
| `.backward()` | Compute all gradients in one pass (call once per forward pass) |
| `torch.no_grad()` | Inference mode — skip building graph, saves memory and speed |

In [ ]:
if TORCH_AVAILABLE:
    # 1. detach() - stops gradient flow
    x = torch.tensor(3.0, requires_grad=True)
    y = x * 2
    y_detached = y.detach()   # y_detached has no grad_fn

    print('y.requires_grad:', y.requires_grad)
    print('y_detached.requires_grad:', y_detached.requires_grad)
    print('y.grad_fn:', y.grad_fn)
    print('y_detached.grad_fn:', y_detached.grad_fn, '(None = detached from graph)')

    # 2. torch.no_grad() - inference mode (no graph built)
    x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
    with torch.no_grad():
        y = x * 2 + 1
    print('\nInside no_grad(): y.requires_grad =', y.requires_grad)

    # 3. Accumulating gradients gotcha
    x = torch.tensor(2.0, requires_grad=True)
    for i in range(3):
        loss = x ** 2
        loss.backward()  # gradients ACCUMULATE without zero_grad!
        print(f'After backward {i+1}: x.grad = {x.grad.item()} (accumulated!)')

    # Correct pattern: zero out gradients before each backward
    x = torch.tensor(2.0, requires_grad=True)
    for i in range(3):
        loss = x ** 2
        if x.grad is not None:
            x.grad.zero_()  # zero out first!
        loss.backward()
        print(f'After zero+backward {i+1}: x.grad = {x.grad.item()} (correct)')
else:
    print('Pattern examples (no torch available):')
    print('detach(): use to stop gradient flow')
    print('no_grad(): use during inference to save memory')
    print('zero_grad(): call before each backward() pass')

### 1.5 GPU vs CPU Tensors — The `.to(device)` Pattern

The standard pattern for device-agnostic code:

In [ ]:
if TORCH_AVAILABLE:
    # Standard pattern
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    # Move tensors to device
    x = torch.randn(3, 3)
    x = x.to(device)
    print(f'Tensor device: {x.device}')

    # Move model to device
    model = nn.Linear(10, 5)
    model = model.to(device)
    print(f'Model device (weight): {next(model.parameters()).device}')

    # Data must be on same device as model!
    batch = torch.randn(4, 10).to(device)
    output = model(batch)
    print(f'Forward pass works: output shape {output.shape}')

    # Common mistake: mixing devices
    try:
        cpu_tensor = torch.randn(4, 10)  # on cpu
        gpu_model = nn.Linear(10, 5).to(device)
        out = gpu_model(cpu_tensor)  # this will error if CUDA is available
        print('Both on CPU, no error (CUDA not available)')
    except RuntimeError as e:
        print(f'Expected error: {str(e)[:80]}')
else:
    print('Device management patterns:')
    print('device = torch.device("cuda" if torch.cuda.is_available() else "cpu")')
    print('model = model.to(device)')
    print('data = data.to(device)')
    print('Rule: model and data must be on the same device')

---
## SECTION 2: Neural Network from Scratch — MLP on MNIST

### 2.1 Load Dataset

In [ ]:
# Load dataset with fallback
if TORCH_AVAILABLE and TORCHVISION_AVAILABLE:
    print('Loading MNIST via torchvision...')
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_dataset = datasets.MNIST('./data', train=True,  download=True, transform=transform)
    test_dataset  = datasets.MNIST('./data', train=False, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=64, shuffle=False)
    N_CLASSES = 10
    INPUT_DIM = 784
    print(f'Train size: {len(train_dataset)}, Test size: {len(test_dataset)}')
    USE_LOADER = True
else:
    print('Using sklearn digits dataset (fallback)...')
    digits = sk_datasets.load_digits()
    X = digits.data.astype(np.float32)
    y = digits.target.astype(np.int64)
    # normalize
    X = X / 16.0
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    N_CLASSES = 10
    INPUT_DIM = 64
    print(f'Train size: {len(X_train)}, Test size: {len(X_test)}, Input dim: {INPUT_DIM}')
    USE_LOADER = False
    if TORCH_AVAILABLE:
        X_train_t = torch.FloatTensor(X_train)
        X_test_t  = torch.FloatTensor(X_test)
        y_train_t = torch.LongTensor(y_train)
        y_test_t  = torch.LongTensor(y_test)
        from torch.utils.data import TensorDataset, DataLoader
        train_ds = TensorDataset(X_train_t, y_train_t)
        test_ds  = TensorDataset(X_test_t,  y_test_t)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)
        USE_LOADER = True

### 2.2 Build the MLP

In [ ]:
if TORCH_AVAILABLE:
    class MLP(nn.Module):
        """Multi-Layer Perceptron for digit classification."""
        def __init__(self, input_dim, hidden_dims, n_classes, dropout_p=0.0):
            super(MLP, self).__init__()
            layers = []
            prev_dim = input_dim
            for h in hidden_dims:
                layers.append(nn.Linear(prev_dim, h))
                layers.append(nn.ReLU())
                if dropout_p > 0:
                    layers.append(nn.Dropout(p=dropout_p))
                prev_dim = h
            layers.append(nn.Linear(prev_dim, n_classes))
            self.network = nn.Sequential(*layers)

        def forward(self, x):
            # Flatten if needed (for image batches)
            if x.dim() > 2:
                x = x.view(x.size(0), -1)
            return self.network(x)

    # Build models
    model_no_dropout = MLP(INPUT_DIM, [256, 128], N_CLASSES, dropout_p=0.0).to(DEVICE)
    model_dropout    = MLP(INPUT_DIM, [256, 128], N_CLASSES, dropout_p=0.5).to(DEVICE)

    print('MLP (no dropout):')
    print(model_no_dropout)
    total_params = sum(p.numel() for p in model_no_dropout.parameters())
    print(f'Total parameters: {total_params:,}')
else:
    print('Defining MLP structure (numpy version):')
    print('Input(', INPUT_DIM, ') -> Linear(256) -> ReLU -> Linear(128) -> ReLU -> Linear(', N_CLASSES, ')')

### 2.3 Full Training Loop

The canonical PyTorch training loop:
1. **Forward pass** — compute predictions
2. **Compute loss** — compare predictions to labels
3. **Zero gradients** — clear old grads (they accumulate!)
4. **Backward pass** — compute gradients via autograd
5. **Optimizer step** — update weights

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=1e-3, device='cpu'):
    """Full training loop. Returns history dict."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        # --- Training phase ---
        model.train()   # sets dropout/batchnorm to training mode
        running_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            # Step 1: Forward pass
            logits = model(batch_X)

            # Step 2: Compute loss
            loss = criterion(logits, batch_y)

            # Step 3: Zero gradients (MUST do before backward)
            optimizer.zero_grad()

            # Step 4: Backward pass
            loss.backward()

            # Step 5: Update weights
            optimizer.step()

            running_loss += loss.item() * batch_X.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_loss)

        # --- Validation phase ---
        model.eval()    # sets dropout/batchnorm to eval mode
        correct = 0
        total = 0
        with torch.no_grad():  # no graph building during eval
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                logits = model(batch_X)
                preds = logits.argmax(dim=1)
                correct += (preds == batch_y).sum().item()
                total += batch_y.size(0)

        val_acc = correct / total
        history['val_acc'].append(val_acc)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:2d}/{epochs} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}')

    return history


if TORCH_AVAILABLE and USE_LOADER:
    print('Training MLP (no dropout)...')
    history_no_drop = train_model(model_no_dropout, train_loader, test_loader, epochs=10, device=DEVICE)
else:
    print('(PyTorch not available - skipping training)')
    history_no_drop = {
        'train_loss': [1.8, 1.2, 0.9, 0.7, 0.55, 0.45, 0.38, 0.33, 0.29, 0.26],
        'val_acc':    [0.55, 0.68, 0.75, 0.80, 0.83, 0.86, 0.87, 0.88, 0.89, 0.90]
    }

### 2.4 Plot Training Curves

In [ ]:
if TORCH_AVAILABLE and USE_LOADER:
    print('Training MLP (with dropout=0.5)...')
    history_dropout = train_model(model_dropout, train_loader, test_loader, epochs=10, device=DEVICE)
else:
    history_dropout = {
        'train_loss': [1.9, 1.4, 1.1, 0.9, 0.75, 0.65, 0.58, 0.53, 0.49, 0.46],
        'val_acc':    [0.52, 0.65, 0.72, 0.77, 0.81, 0.84, 0.86, 0.87, 0.88, 0.89]
    }

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_no_drop['train_loss'],  label='No Dropout', color='steelblue')
axes[0].plot(history_dropout['train_loss'],  label='Dropout=0.5', color='tomato', linestyle='--')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_no_drop['val_acc'],  label='No Dropout', color='steelblue')
axes[1].plot(history_dropout['val_acc'],  label='Dropout=0.5', color='tomato', linestyle='--')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mlp_training_curves.png', dpi=80, bbox_inches='tight')
plt.show()
print('Dropout slows convergence but improves generalization (val accuracy)')

### 2.5 Dropout Effect on Overfitting

Dropout randomly zeros neurons during training. This:
1. Prevents co-adaptation of neurons
2. Acts like training an ensemble of subnetworks
3. Forces each neuron to learn useful features independently

In [ ]:
if TORCH_AVAILABLE:
    # Show dropout behavior: train vs eval mode
    drop_layer = nn.Dropout(p=0.5)
    x = torch.ones(1, 10)

    drop_layer.train()
    out_train = drop_layer(x)
    print('Dropout in TRAIN mode (zeroes ~50% of values):')
    print(out_train)
    print('Non-zero count:', (out_train != 0).sum().item(), '/ 10')

    drop_layer.eval()
    out_eval = drop_layer(x)
    print('\nDropout in EVAL mode (passes all values unchanged):')
    print(out_eval)
    print('Non-zero count:', (out_eval != 0).sum().item(), '/ 10')

    print('\nKey insight: During training, non-zero outputs are SCALED UP by 1/(1-p)')
    print(f'With p=0.5, active values become 2x to maintain expected sum')
else:
    print('Dropout explanation:')
    print('- During training: each neuron output set to 0 with probability p')
    print('- Active outputs scaled by 1/(1-p) to maintain expected value')
    print('- During inference: dropout is disabled, all neurons active')
    print('- Effect: prevents co-adaptation, acts as ensemble')

---
## SECTION 3: Convolutional Neural Networks

### 3.1 Convolution Intuition — 5x5 Example

A convolution filter slides over the input image, computing a dot product at each position. This detects local patterns (edges, textures).

In [ ]:
# Manual convolution demonstration (numpy - no torch needed)

def manual_conv2d(input_map, kernel, stride=1, padding=0):
    """2D convolution from scratch."""
    H, W = input_map.shape
    KH, KW = kernel.shape
    if padding > 0:
        input_map = np.pad(input_map, padding, mode='constant')
        H, W = input_map.shape
    out_H = (H - KH) // stride + 1
    out_W = (W - KW) // stride + 1
    output = np.zeros((out_H, out_W))
    for i in range(0, out_H):
        for j in range(0, out_W):
            patch = input_map[i*stride:i*stride+KH, j*stride:j*stride+KW]
            output[i, j] = np.sum(patch * kernel)
    return output

# 5x5 input with a vertical edge
input_img = np.array([
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 1]
], dtype=float)

# Sobel vertical edge detector
vertical_edge_kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=float)

feature_map = manual_conv2d(input_img, vertical_edge_kernel)

print('Input (5x5) — vertical edge at column 2:')
print(input_img.astype(int))
print()
print('Vertical Edge Kernel (3x3):')
print(vertical_edge_kernel.astype(int))
print()
print('Feature Map (3x3) — high values = edge detected:')
print(feature_map)
print()
print('The large values (4, 8) appear at column 1 where the vertical edge is!')

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(input_img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input (5x5)')
axes[0].axis('off')
axes[1].imshow(vertical_edge_kernel, cmap='RdBu')
axes[1].set_title('Kernel (3x3)')
axes[1].axis('off')
axes[2].imshow(feature_map, cmap='hot')
axes[2].set_title('Feature Map (3x3)')
axes[2].axis('off')
plt.tight_layout()
plt.savefig('conv_demo.png', dpi=80, bbox_inches='tight')
plt.show()

### 3.2 CNN Architecture

Conv layers learn filters automatically during training. The architecture:
- Conv layer: learns spatial features (edges, textures, shapes)
- MaxPool: reduces spatial size, adds translation invariance
- FC layers: combine features for classification

In [ ]:
if TORCH_AVAILABLE:
    class CNN(nn.Module):
        """Small CNN for image classification."""
        def __init__(self, in_channels, n_classes, img_size=8):
            super(CNN, self).__init__()
            # Conv block 1
            self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
            self.bn1   = nn.BatchNorm2d(32)
            # Conv block 2
            self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
            self.bn2   = nn.BatchNorm2d(64)
            self.pool  = nn.MaxPool2d(2, 2)
            self.dropout = nn.Dropout(0.25)

            # Compute flattened size
            reduced = img_size // 2  # after one pool
            self.fc1 = nn.Linear(64 * reduced * reduced, 128)
            self.fc2 = nn.Linear(128, n_classes)

        def forward(self, x):
            # Block 1: conv -> bn -> relu
            x = F.relu(self.bn1(self.conv1(x)))
            # Block 2: conv -> bn -> relu -> pool
            x = self.pool(F.relu(self.bn2(self.conv2(x))))
            x = self.dropout(x)
            # Classifier head
            x = x.view(x.size(0), -1)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return x

    # For sklearn digits (8x8 images, 1 channel)
    cnn_model = CNN(in_channels=1, n_classes=10, img_size=8).to(DEVICE)
    print('CNN Architecture:')
    print(cnn_model)
    total = sum(p.numel() for p in cnn_model.parameters())
    print(f'\nTotal parameters: {total:,}')

    # Show feature map shapes
    dummy = torch.randn(1, 1, 8, 8).to(DEVICE)
    print('\nFeature map dimensions through network:')
    print(f'Input:          {dummy.shape}')
    x = dummy
    x = F.relu(cnn_model.bn1(cnn_model.conv1(x)))
    print(f'After conv1+bn: {x.shape}   (32 feature maps)')
    x = cnn_model.pool(F.relu(cnn_model.bn2(cnn_model.conv2(x))))
    print(f'After conv2+pool: {x.shape} (64 feature maps, halved spatially)')
    x = x.view(x.size(0), -1)
    print(f'Flattened:      {x.shape}')
else:
    print('CNN structure (no torch available):')
    print('Input -> Conv(32) -> BN -> ReLU -> Conv(64) -> BN -> ReLU -> MaxPool -> FC(128) -> FC(10)')

### 3.3 Train CNN and Visualize Feature Maps

In [ ]:
if TORCH_AVAILABLE and USE_LOADER:
    # Prepare CNN data: reshape digits to (N, 1, 8, 8)
    if not TORCHVISION_AVAILABLE:
        # sklearn digits: reshape to image format
        from torch.utils.data import TensorDataset, DataLoader
        X_cnn_train = torch.FloatTensor(X_train).reshape(-1, 1, 8, 8)
        X_cnn_test  = torch.FloatTensor(X_test).reshape(-1, 1, 8, 8)
        y_cnn_train = torch.LongTensor(y_train)
        y_cnn_test  = torch.LongTensor(y_test)
        cnn_train_loader = DataLoader(TensorDataset(X_cnn_train, y_cnn_train), batch_size=32, shuffle=True)
        cnn_test_loader  = DataLoader(TensorDataset(X_cnn_test,  y_cnn_test),  batch_size=32, shuffle=False)
        IMG_SIZE = 8
        IN_CH = 1
    else:
        # MNIST: 28x28 images
        cnn_train_loader = train_loader
        cnn_test_loader  = test_loader
        IMG_SIZE = 28
        IN_CH = 1
        cnn_model = CNN(in_channels=IN_CH, n_classes=10, img_size=IMG_SIZE).to(DEVICE)

    print('Training CNN...')
    cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    cnn_history = {'train_loss': [], 'val_acc': []}

    for epoch in range(10):
        cnn_model.train()
        running_loss = 0.0
        for bx, by in cnn_train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            logits = cnn_model(bx)
            loss = criterion(logits, by)
            cnn_optimizer.zero_grad()
            loss.backward()
            cnn_optimizer.step()
            running_loss += loss.item() * bx.size(0)
        epoch_loss = running_loss / len(cnn_train_loader.dataset)

        cnn_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for bx, by in cnn_test_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                preds = cnn_model(bx).argmax(dim=1)
                correct += (preds == by).sum().item()
                total += by.size(0)
        val_acc = correct / total
        cnn_history['train_loss'].append(epoch_loss)
        cnn_history['val_acc'].append(val_acc)
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:2d}/10 | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}')
else:
    print('(skipping CNN training - no torch/data)')
    cnn_history = {
        'train_loss': [1.5, 0.9, 0.6, 0.4, 0.3, 0.25, 0.22, 0.19, 0.17, 0.15],
        'val_acc':    [0.65, 0.78, 0.85, 0.89, 0.91, 0.92, 0.93, 0.94, 0.94, 0.95]
    }

In [ ]:
# Visualize feature maps from conv1
if TORCH_AVAILABLE and USE_LOADER:
    cnn_model.eval()
    with torch.no_grad():
        if not TORCHVISION_AVAILABLE:
            sample = X_cnn_test[:1].to(DEVICE)
        else:
            sample = next(iter(cnn_test_loader))[0][:1].to(DEVICE)

        # Get feature maps after conv1
        feat_maps = F.relu(cnn_model.bn1(cnn_model.conv1(sample)))
        feat_maps = feat_maps.squeeze(0).cpu().numpy()  # (32, H, W)

    n_show = min(16, feat_maps.shape[0])
    fig, axes = plt.subplots(2, 8, figsize=(14, 4))
    fig.suptitle('Feature Maps After Conv1 (what filters detect)', fontsize=12)
    for i, ax in enumerate(axes.flat):
        if i < n_show:
            ax.imshow(feat_maps[i], cmap='viridis')
            ax.set_title(f'F{i}', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('feature_maps.png', dpi=80, bbox_inches='tight')
    plt.show()
    print('Each feature map responds to different patterns in the input image')
else:
    print('Feature map visualization requires PyTorch')
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.text(0.5, 0.5, 'Feature maps show what\neach filter detects\n(edges, curves, textures)',
            ha='center', va='center', fontsize=14, transform=ax.transAxes)
    ax.axis('off')
    plt.savefig('feature_maps_placeholder.png', dpi=80)
    plt.show()

### 3.4 Transfer Learning with Pretrained ResNet

Transfer learning strategy:
1. Load pretrained model (weights from ImageNet)
2. Freeze all layers (no gradient updates)
3. Replace final classifier with task-specific head
4. Fine-tune: either just the head, or unfreeze later layers too

In [ ]:
if TORCH_AVAILABLE and TORCHVISION_AVAILABLE:
    from torchvision import models

    # Load pretrained ResNet18
    resnet = models.resnet18(pretrained=False)  # pretrained=True in real use
    print('ResNet18 Architecture (abbreviated):')
    print(f'  layer1: {resnet.layer1}')
    print(f'  ...several residual blocks...')
    print(f'  fc: {resnet.fc}')

    # Strategy 1: Freeze everything, replace final FC
    for param in resnet.parameters():
        param.requires_grad = False
    print('\nAfter freezing all layers:')
    trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
    total = sum(p.numel() for p in resnet.parameters())
    print(f'  Trainable: {trainable:,} / {total:,}')

    # Replace classifier for our task (e.g., 10 classes)
    num_features = resnet.fc.in_features
    resnet.fc = nn.Linear(num_features, 10)

    print('\nAfter replacing FC head:')
    trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
    print(f'  Trainable: {trainable:,} / {total:,}  (only new FC layer)')

    # Strategy 2: Unfreeze last layer for fine-tuning
    for param in resnet.layer4.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
    print(f'  After unfreezing layer4: {trainable:,} trainable params')

    print('\nTransfer learning rule of thumb:')
    print('  Small dataset, similar domain -> freeze all, train head only')
    print('  Large dataset, different domain -> unfreeze more layers')
else:
    print('Transfer Learning Pattern (no torchvision):')
    print()
    print('# 1. Load pretrained model')
    print('model = models.resnet18(pretrained=True)')
    print()
    print('# 2. Freeze all layers')
    print('for param in model.parameters():')
    print('    param.requires_grad = False')
    print()
    print('# 3. Replace classifier head')
    print('model.fc = nn.Linear(model.fc.in_features, num_classes)')
    print()
    print('# 4. Only FC parameters are updated')
    print('optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)')

---
## SECTION 4: Training Tricks

### 4.1 Learning Rate Scheduling

In [ ]:
if TORCH_AVAILABLE:
    # Visualize different LR schedules
    dummy_model = nn.Linear(10, 1)
    base_lr = 0.1
    epochs = 30

    # StepLR: multiply LR by gamma every step_size epochs
    opt1 = optim.SGD(dummy_model.parameters(), lr=base_lr)
    sched1 = optim.lr_scheduler.StepLR(opt1, step_size=10, gamma=0.1)

    # CosineAnnealingLR: smooth cosine decay
    opt2 = optim.SGD(nn.Linear(10,1).parameters(), lr=base_lr)
    sched2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs)

    # ExponentialLR
    opt3 = optim.SGD(nn.Linear(10,1).parameters(), lr=base_lr)
    sched3 = optim.lr_scheduler.ExponentialLR(opt3, gamma=0.9)

    lrs1, lrs2, lrs3 = [], [], []
    for _ in range(epochs):
        lrs1.append(opt1.param_groups[0]['lr'])
        lrs2.append(opt2.param_groups[0]['lr'])
        lrs3.append(opt3.param_groups[0]['lr'])
        sched1.step()
        sched2.step()
        sched3.step()
else:
    epochs = 30
    lrs1 = [0.1 if i < 10 else (0.01 if i < 20 else 0.001) for i in range(epochs)]
    lrs2 = [0.1 * (1 + np.cos(np.pi * i / epochs)) / 2 for i in range(epochs)]
    lrs3 = [0.1 * (0.9 ** i) for i in range(epochs)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lrs1, label='StepLR (step=10, gamma=0.1)', marker='o', markersize=3)
ax.plot(lrs2, label='CosineAnnealingLR', marker='s', markersize=3)
ax.plot(lrs3, label='ExponentialLR (gamma=0.9)', marker='^', markersize=3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('lr_schedules.png', dpi=80, bbox_inches='tight')
plt.show()
print('CosineAnnealing is the most common in modern practice (smooth, no sharp drops)')

### 4.2 Batch Normalization — Why It Helps

In [ ]:
if TORCH_AVAILABLE:
    # Demonstrate what BatchNorm does to activations
    torch.manual_seed(42)

    # Simulate activations before BN
    activations = torch.randn(64, 32) * 10 + 5  # mean=5, std=10
    print('Before BatchNorm:')
    print(f'  mean: {activations.mean().item():.3f}, std: {activations.std().item():.3f}')

    bn = nn.BatchNorm1d(32)
    bn.eval()

    # Manually normalize (what BN does during training)
    mean = activations.mean(dim=0)
    std  = activations.std(dim=0)
    norm = (activations - mean) / (std + 1e-5)
    print('\nAfter normalization (mean=0, std=1):')
    print(f'  mean: {norm.mean().item():.3f}, std: {norm.std().item():.3f}')

    print()
    print('Why BatchNorm helps:')
    print('  1. Reduces internal covariate shift (activations stay in good range)')
    print('  2. Acts as regularizer (slight noise from batch statistics)')
    print('  3. Allows higher learning rates (more stable training)')
    print('  4. gamma/beta are learnable -> can represent identity if needed')
    print()
    print('BatchNorm has learnable parameters:')
    bn2 = nn.BatchNorm1d(32)
    for name, p in bn2.named_parameters():
        print(f'  {name}: shape {p.shape} (gamma=weight, beta=bias)')
else:
    activations = np.random.randn(64, 32) * 10 + 5
    mean = activations.mean(axis=0)
    std  = activations.std(axis=0)
    norm = (activations - mean) / (std + 1e-8)
    print('Before BN: mean={:.2f}, std={:.2f}'.format(activations.mean(), activations.std()))
    print('After BN:  mean={:.2f}, std={:.2f}'.format(norm.mean(), norm.std()))

### 4.3 Gradient Clipping — When and Why

Gradient clipping prevents **exploding gradients** in:
- RNNs/LSTMs (long sequences, many multiplications)
- Very deep networks
- When loss landscape has sharp cliffs

In [ ]:
if TORCH_AVAILABLE:
    model_gc = nn.Linear(10, 1)
    optimizer_gc = optim.SGD(model_gc.parameters(), lr=0.1)

    # Simulate large gradients
    x = torch.randn(5, 10) * 100  # large input
    y = torch.randn(5, 1)
    loss = F.mse_loss(model_gc(x), y)
    loss.backward()

    # Check gradient norm before clipping
    grad_norm_before = 0
    for p in model_gc.parameters():
        if p.grad is not None:
            grad_norm_before += p.grad.data.norm(2).item() ** 2
    grad_norm_before = grad_norm_before ** 0.5
    print(f'Gradient norm BEFORE clipping: {grad_norm_before:.4f}')

    # Clip gradients to max_norm=1.0
    torch.nn.utils.clip_grad_norm_(model_gc.parameters(), max_norm=1.0)

    grad_norm_after = 0
    for p in model_gc.parameters():
        if p.grad is not None:
            grad_norm_after += p.grad.data.norm(2).item() ** 2
    grad_norm_after = grad_norm_after ** 0.5
    print(f'Gradient norm AFTER clipping:  {grad_norm_after:.4f}')
    print(f'Clipping active: {grad_norm_before > 1.0}')

    print()
    print('Standard training loop with gradient clipping:')
    print('  optimizer.zero_grad()')
    print('  loss.backward()')
    print('  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ADD THIS')
    print('  optimizer.step()')
else:
    print('Gradient clipping pseudo-code:')
    print('  grad_norm = sqrt(sum(g^2 for all params))')
    print('  if grad_norm > max_norm: scale = max_norm / grad_norm')
    print('  Each gradient *= scale')
    print('  This preserves direction but limits magnitude')

### 4.4 Early Stopping Implementation

In [ ]:
class EarlyStopping:
    """Stop training when validation metric stops improving."""
    def __init__(self, patience=5, min_delta=0.001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_value = float('inf') if mode == 'min' else float('-inf')
        self.counter = 0
        self.best_epoch = 0

    def step(self, value, epoch):
        """
        Returns True if training should stop.
        """
        if self.mode == 'min':
            improved = value < self.best_value - self.min_delta
        else:
            improved = value > self.best_value + self.min_delta

        if improved:
            self.best_value = value
            self.counter = 0
            self.best_epoch = epoch
        else:
            self.counter += 1

        return self.counter >= self.patience


# Simulate validation loss
val_losses = [0.8, 0.6, 0.5, 0.48, 0.47, 0.47, 0.47, 0.47, 0.47, 0.47]

es = EarlyStopping(patience=3, min_delta=0.005, mode='min')
print('Epoch | Val Loss | Counter | Stop?')
print('-' * 40)
for epoch, vl in enumerate(val_losses):
    should_stop = es.step(vl, epoch)
    print(f'  {epoch+1:2d}  |  {vl:.3f}   |   {es.counter}     | {"STOP" if should_stop else "continue"}')
    if should_stop:
        print(f'\nEarly stopping at epoch {epoch+1}. Best was epoch {es.best_epoch+1} with loss {es.best_value:.4f}')
        break

### 4.5 Weight Initialization Strategies

Bad initialization -> vanishing or exploding gradients from the start. Key strategies:

| Activation | Initialization | Formula |
|------------|---------------|----------|
| ReLU | He (Kaiming) | std = sqrt(2/fan_in) |
| Tanh/Sigmoid | Xavier/Glorot | std = sqrt(2/(fan_in+fan_out)) |
| Linear | Kaiming Uniform | default in PyTorch |

In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(42)

    def check_activations(init_fn, activation, n_layers=10, n_neurons=256):
        """Track activation statistics through a deep network."""
        x = torch.randn(100, n_neurons)
        means, stds = [], []

        for _ in range(n_layers):
            W = torch.empty(n_neurons, n_neurons)
            init_fn(W)
            x = activation(x @ W.T)
            means.append(x.mean().item())
            stds.append(x.std().item())
        return means, stds

    # Naive initialization (too small -> vanishing)
    means_bad, stds_bad = check_activations(
        lambda W: nn.init.normal_(W, std=0.01),
        torch.relu
    )

    # He initialization (correct for ReLU)
    means_he, stds_he = check_activations(
        lambda W: nn.init.kaiming_normal_(W, nonlinearity='relu'),
        torch.relu
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    layers = range(1, 11)

    axes[0].plot(layers, stds_bad, 'tomato', label='Normal(0, 0.01)', marker='o')
    axes[0].plot(layers, stds_he, 'steelblue', label='He Init', marker='s')
    axes[0].set_title('Activation Std Through 10 Layers')
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('Std of activations')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(layers, means_bad, 'tomato', label='Normal(0, 0.01)', marker='o')
    axes[1].plot(layers, means_he, 'steelblue', label='He Init', marker='s')
    axes[1].set_title('Activation Mean Through 10 Layers')
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Mean of activations')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('weight_init.png', dpi=80, bbox_inches='tight')
    plt.show()
    print('Bad init: activations collapse to 0 (vanishing gradient)')
    print('He init: activations remain stable across all layers')
else:
    print('Weight initialization strategies:')
    print()
    print('Xavier/Glorot: std = sqrt(2 / (fan_in + fan_out))')
    print('  Use with: tanh, sigmoid')
    print()
    print('He/Kaiming:   std = sqrt(2 / fan_in)')
    print('  Use with: ReLU, LeakyReLU')
    print()
    print('Why it matters:')
    print('  Too small -> activations shrink layer by layer (vanishing gradient)')
    print('  Too large -> activations explode layer by layer (NaN loss)')

---
## SECTION 5: 15 Deep Learning Interview Questions

### Q1: What is the vanishing gradient problem and how do you fix it?

In [ ]:
print('='*60)
print('Q1: Vanishing Gradient Problem')
print('='*60)
print()
print('PLAIN ENGLISH:')
print('  In deep networks, gradients are multiplied together as they')
print('  flow backwards through layers (chain rule). If each gradient')
print('  is < 1, multiplying many of them makes the gradient')
print('  exponentially small. Early layers learn nothing.')
print()
print('MATH:')
print('  grad_layer_1 = grad_output * W_n * W_{n-1} * ... * W_1')
print('  If |W_i| < 1 and sigmoid saturation -> gradient -> 0')
print()
print('FIXES:')
print('  1. Use ReLU instead of sigmoid (gradient = 1 for positive inputs)')
print('  2. Use residual connections (skip connections in ResNet)')
print('  3. Use Batch Normalization (keeps activations in good range)')
print('  4. Use LSTM/GRU in RNNs (gate controls gradient flow)')
print('  5. Use gradient clipping')
print('  6. Use He/Xavier initialization')

# Code proof: sigmoid vs ReLU gradient
if TORCH_AVAILABLE:
    def sigmoid_grad(x):
        s = torch.sigmoid(x)
        return s * (1 - s)

    x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0, 5.0])
    sig_grads = sigmoid_grad(x)
    relu_grads = (x > 0).float()

    print()
    print('Gradient comparison (sigmoid vs ReLU):')
    print(f'x:              {x.tolist()}')
    print(f'sigmoid grad:   {[round(g, 4) for g in sig_grads.tolist()]}')
    print(f'ReLU grad:      {relu_grads.tolist()}')
    print('Sigmoid saturates at extremes (near 0), ReLU does not!')
else:
    x = np.array([-3.0, -1.0, 0.0, 1.0, 3.0, 5.0])
    sig_grad = lambda x: 1/(1+np.exp(-x)) * (1 - 1/(1+np.exp(-x)))
    print('\nSigmoid gradients:', [round(g, 4) for g in sig_grad(x).tolist()])
    print('ReLU gradients:   ', [1.0 if v > 0 else 0.0 for v in x])
    print('Sigmoid saturates near 0 at extremes!')

### Q2: Why does Batch Normalization work?

In [ ]:
print('='*60)
print('Q2: Why Does Batch Normalization Work?')
print('='*60)
print()
print('PLAIN ENGLISH:')
print('  Without BN, each layer sees a shifting distribution of inputs')
print('  as earlier layers update ("internal covariate shift").')
print('  BN normalizes each layer input to mean=0, std=1 per batch,')
print('  so each layer trains in a stable, consistent environment.')
print()
print('THE FORMULA (per feature, per batch):')
print('  x_hat = (x - batch_mean) / sqrt(batch_var + eps)')
print('  y = gamma * x_hat + beta  (learnable scale and shift)')
print()
print('WHY IT HELPS:')
print('  1. Reduces internal covariate shift')
print('  2. Acts as regularizer (adds noise via batch statistics)')
print('  3. Allows higher learning rates (more stable gradient flow)')
print('  4. Reduces dependence on careful initialization')
print()
print('KEY INSIGHT: gamma and beta are learnable!')
print('  If BN is not useful for a layer, it can learn gamma=std, beta=mean')
print('  to recover the identity transform. So BN never hurts, may help.')
print()
print('TRAIN vs EVAL difference:')
print('  Train: normalize using current BATCH statistics')
print('  Eval:  normalize using RUNNING statistics (accumulated during training)')
print('  --> Always call model.eval() during inference!')

### Q3: When would you use ReLU vs Sigmoid vs Tanh?

In [ ]:
# Visualize the activations and their gradients
x = np.linspace(-4, 4, 200)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)
leaky_relu = np.where(x > 0, x, 0.01 * x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Activation functions
axes[0].plot(x, sigmoid,    label='Sigmoid',    color='royalblue')
axes[0].plot(x, tanh,       label='Tanh',       color='tomato')
axes[0].plot(x, relu,       label='ReLU',       color='forestgreen')
axes[0].plot(x, leaky_relu, label='LeakyReLU',  color='orange', linestyle='--')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_title('Activation Functions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('x')

# Gradients
sig_grad = sigmoid * (1 - sigmoid)
tanh_grad = 1 - tanh**2
relu_grad = (x > 0).astype(float)
lr_grad = np.where(x > 0, 1.0, 0.01)
axes[1].plot(x, sig_grad,  label='Sigmoid grad',   color='royalblue')
axes[1].plot(x, tanh_grad, label='Tanh grad',      color='tomato')
axes[1].plot(x, relu_grad, label='ReLU grad',      color='forestgreen')
axes[1].plot(x, lr_grad,   label='LeakyReLU grad', color='orange', linestyle='--')
axes[1].set_title('Gradients (dactivation/dx)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('x')
axes[1].set_ylabel('gradient')

plt.tight_layout()
plt.savefig('activations.png', dpi=80, bbox_inches='tight')
plt.show()

print('When to use each:')
print('  ReLU:       Hidden layers (default choice). Fast, avoids saturation.')
print('  LeakyReLU:  If dying ReLU is a problem (many dead neurons).')
print('  Tanh:       Zero-centered (better than sigmoid). Used in LSTMs.')
print('  Sigmoid:    Output layer for BINARY classification only.')
print('  Softmax:    Output layer for MULTI-CLASS classification.')
print()
print('Dying ReLU problem: if a neuron gets large negative input,')
print('gradient = 0 forever -> neuron never recovers. Fix: LeakyReLU.')

### Q4: What is the difference between CNN and fully connected layers?

In [ ]:
print('='*60)
print('Q4: CNN vs Fully Connected Layers')
print('='*60)
print()
print('PARAMETER COUNT COMPARISON')
print('Input: 28x28 image (784 pixels), 32 filters, 3x3 kernel')
print()

# Fully connected
fc_params = 784 * 32  # each output connected to all inputs
print(f'Fully Connected (784 -> 32 outputs):')
print(f'  Params: {fc_params:,}  (each output sees all 784 pixels)')

# Conv layer
conv_params = 3 * 3 * 1 * 32  # kernel_h * kernel_w * in_channels * out_channels
print(f'Conv2d(in=1, out=32, kernel=3x3):')
print(f'  Params: {conv_params:,}  (each filter is 9 weights, SHARED across all positions)')
print(f'  Reduction factor: {fc_params // conv_params}x fewer parameters!')
print()
print('KEY DIFFERENCES:')
print('  Property          | Conv              | FC')
print('  ------------------|-------------------|-----------')
print('  Weight sharing    | YES (across space)| NO')
print('  Local connectivity| YES (kernel size) | NO (all-to-all)')
print('  Translation inv.  | YES               | NO')
print('  Inductive bias    | Spatial structure | None')
print('  Good for          | Images, sequences | Tabular data, final layer')
print()
print('INDUCTIVE BIAS of CNN:')
print('  Assumption: nearby pixels are related, features recur across image.')
print('  This is true for images! FC would need to relearn same edge')
print('  detector 784 times (once per position).')

### Q5: How does dropout prevent overfitting?

In [ ]:
print('='*60)
print('Q5: How Dropout Prevents Overfitting')
print('='*60)
print()
print('MECHANISM:')
print('  During training: each neuron is zeroed with probability p')
print('  Network must learn redundant representations')
print('  No single neuron can specialize too much')
print()
print('FOUR INTERPRETATIONS:')
print()
print('  1. Ensemble method:')
print('     Each training step trains a different subnetwork (2^n possible)')
print('     At test time, average all subnetworks by using full network with scaled weights')
print()
print('  2. Prevents co-adaptation:')
print('     Neurons cant learn to "vote together" because any voter may be absent')
print('     Each neuron must be individually useful')
print()
print('  3. Regularization:')
print('     Adds noise -> prevents memorizing training data')
print('     Similar effect to L2 regularization in some analyses')
print()
print('  4. Feature robustness:')
print('     Network learns features that are robust to partial information loss')
print()
print('PRACTICAL TIPS:')
print('  p=0.5 for hidden layers (most common)')
print('  p=0.1-0.2 for input layer (dont kill too much input)')
print('  p=0.0 (no dropout) for small datasets (not enough data to regularize)')
print('  Always: model.train() during training, model.eval() during inference!')

if TORCH_AVAILABLE:
    # Show inverted dropout scaling
    p = 0.5
    x = torch.ones(1000)
    drop = nn.Dropout(p=p)
    drop.train()
    out = drop(x)
    print(f'\nInverted dropout proof (p={p}):')
    print(f'  Input mean: {x.mean().item():.3f}')
    print(f'  Output mean after dropout: {out.mean().item():.3f}  (close to 1.0)')
    print(f'  Active values: {(out > 0).sum().item()}/1000')
    print(f'  Active values scale: {out[out>0].mean().item():.3f} (= 1/(1-p) = {1/(1-p):.1f})')

### Q6-Q10: Core Concepts

In [ ]:
print('='*60)
print('Q6: What is the difference between SGD and Adam?')
print('='*60)
print()
print('SGD (Stochastic Gradient Descent):')
print('  w = w - lr * gradient')
print('  Simple, works well with LR scheduling')
print('  Sensitive to learning rate, slow on sparse gradients')
print()
print('Momentum SGD:')
print('  velocity = momentum * velocity - lr * gradient')
print('  w = w + velocity')
print('  Smooths oscillations, accelerates in consistent direction')
print()
print('Adam (Adaptive Moment Estimation):')
print('  m = beta1 * m + (1-beta1) * gradient        # 1st moment (mean)')
print('  v = beta2 * v + (1-beta2) * gradient^2     # 2nd moment (variance)')
print('  m_hat = m / (1 - beta1^t)                   # bias correction')
print('  v_hat = v / (1 - beta2^t)')
print('  w = w - lr * m_hat / (sqrt(v_hat) + eps)')
print()
print('Adam pros: per-parameter adaptive LR, fast convergence, less LR tuning')
print('Adam cons: may not generalize as well as SGD+momentum in some cases')
print('Rule: start with Adam, switch to SGD+momentum for final fine-tuning')

In [ ]:
print('='*60)
print('Q7: What is weight decay / L2 regularization?')
print('='*60)
print()
print('L2 regularization adds a penalty to the loss:')
print('  Loss_total = Loss_data + lambda * sum(w^2)')
print()
print('Effect on gradient update:')
print('  dLoss/dw += 2 * lambda * w')
print('  w_new = w - lr * (grad + 2*lambda*w)')
print('        = w * (1 - 2*lr*lambda) - lr * grad')
print()
print('This DECAYS weights toward 0 every step -> "weight decay"')
print()
print('In PyTorch: weight_decay parameter in optimizer')
print('  optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)')
print()
print('L1 vs L2:')
print('  L1 (lasso): promotes SPARSITY (some weights become exactly 0)')
print('  L2 (ridge): shrinks all weights uniformly toward 0')
print('  Elasticnet: both L1 and L2')

if TORCH_AVAILABLE:
    # Show how weight_decay affects optimizer
    m = nn.Linear(10, 5)
    opt_no_wd  = optim.Adam(m.parameters(), lr=0.01, weight_decay=0)
    opt_with_wd = optim.Adam(m.parameters(), lr=0.01, weight_decay=1e-4)
    print()
    print(f'Adam without weight_decay: {opt_no_wd.param_groups[0]["weight_decay"]}')
    print(f'Adam with weight_decay=1e-4: {opt_with_wd.param_groups[0]["weight_decay"]}')

In [ ]:
print('='*60)
print('Q8: What is a residual connection and why does it help?')
print('='*60)
print()
print('Regular layer:   y = F(x)')
print('Residual block:  y = F(x) + x   (skip connection)')
print()
print('DIAGRAM:')
print('  x --->[Layer]---> F(x)')
print('  |                  |')
print('  +------------------+')
print('           |')
print('        y = F(x) + x')
print()
print('WHY IT HELPS:')
print('  1. Gradient highway: gradients flow directly through skip connection')
print('     dL/dx = dL/dy * (dF/dx + I)  <- identity always provides gradient')
print('  2. Easier to learn identity: if F(x)=0, output = x (no-op layer)')
print('  3. Enables very deep networks (ResNet has 50-152 layers)')
print()
print('CODE EXAMPLE:')
if TORCH_AVAILABLE:
    class ResidualBlock(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.block = nn.Sequential(
                nn.Linear(dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Linear(dim, dim),
                nn.BatchNorm1d(dim)
            )
        def forward(self, x):
            return F.relu(self.block(x) + x)  # skip connection
    print('  See ResidualBlock class above')
    rb = ResidualBlock(64)
    x_test = torch.randn(8, 64)
    print(f'  Input shape: {x_test.shape}, Output shape: {rb(x_test).shape}')
else:
    print('  def forward(self, x):')
    print('      return F.relu(self.block(x) + x)  # skip connection')

In [ ]:
print('='*60)
print('Q9: Explain cross-entropy loss for classification')
print('='*60)
print()
print('Binary cross-entropy:')
print('  L = -[y*log(p) + (1-y)*log(1-p)]')
print('  y: true label (0 or 1), p: predicted probability')
print()
print('Categorical cross-entropy (multi-class):')
print('  L = -sum_c [ y_c * log(p_c) ]')
print('  For one-hot labels: reduces to -log(p_true_class)')
print()
print('INTUITION:')
print('  log(1) = 0   -> perfect prediction, zero loss')
print('  log(0.5) = -0.69 -> moderate confidence, moderate loss')
print('  log(0.01) = -4.6 -> very wrong, high loss')
print()

probs = np.array([0.01, 0.1, 0.5, 0.9, 0.99])
losses = -np.log(probs)
print('Predicted probability | Cross-entropy loss')
print('-' * 40)
for p, l in zip(probs, losses):
    print(f'       {p:.2f}             |     {l:.4f}')

print()
print('PyTorch usage:')
print('  nn.CrossEntropyLoss() expects RAW LOGITS (not softmax output!)')
print('  It applies log_softmax internally for numerical stability')
print('  Do NOT apply softmax before CrossEntropyLoss!')

In [ ]:
print('='*60)
print('Q10: What is the difference between model.train() and model.eval()?')
print('='*60)
print()
print('Layers affected by train/eval mode:')
print('  1. Dropout: active during train, disabled during eval')
print('  2. BatchNorm: uses batch statistics in train, running stats in eval')
print()
print('Failure modes if you forget:')
print('  Forget model.eval(): different results each inference call (dropout noise)')
print('  Forget model.train(): dropout/BN frozen, training may be worse')
print()
if TORCH_AVAILABLE:
    model_demo = nn.Sequential(
        nn.Linear(10, 20),
        nn.Dropout(0.5),
        nn.Linear(20, 5)
    )
    x_demo = torch.randn(3, 10)

    model_demo.train()
    out_train1 = model_demo(x_demo).detach()
    out_train2 = model_demo(x_demo).detach()

    model_demo.eval()
    out_eval1 = model_demo(x_demo).detach()
    out_eval2 = model_demo(x_demo).detach()

    print('Training mode: two forward passes give DIFFERENT results (dropout noise):')
    print(f'  pass1[0]: {out_train1[0, :3].tolist()}')
    print(f'  pass2[0]: {out_train2[0, :3].tolist()}')
    print()
    print('Eval mode: two forward passes give IDENTICAL results (deterministic):')
    print(f'  pass1[0]: {out_eval1[0, :3].tolist()}')
    print(f'  pass2[0]: {out_eval2[0, :3].tolist()}')
else:
    print('model.train() -> dropout active, batch stats for BN')
    print('model.eval()  -> dropout off, running stats for BN (deterministic)')

### Q11-Q15: Advanced Topics

In [ ]:
print('='*60)
print('Q11: What is the difference between L1 and L2 loss?')
print('='*60)
print()
print('L1 loss (MAE): L = mean(|y_pred - y_true|)')
print('L2 loss (MSE): L = mean((y_pred - y_true)^2)')
print()
print('KEY DIFFERENCES:')
errors = np.array([0.1, 0.5, 1.0, 2.0, 5.0, 10.0])
print(f'{"Error":>10} | {"L1":>8} | {"L2":>10}')
print('-' * 35)
for e in errors:
    print(f'{e:>10.1f} | {e:>8.3f} | {e**2:>10.3f}')
print()
print('L2 penalizes LARGE errors much more (squared)')
print('L1 is ROBUST to outliers (linear penalty)')
print()
print('When to use:')
print('  L2 (MSE): standard regression, assume Gaussian noise')
print('  L1 (MAE): robust regression, outliers expected, median-like')
print('  Huber: combines both (L2 for small errors, L1 for large errors)')
print()
print('L1 also promotes SPARSITY in gradients (may lead to 0-gradient for wrong preds)')

In [ ]:
print('='*60)
print('Q12: What happens if learning rate is too high or too low?')
print('='*60)
print()
lr_values = [0.001, 0.01, 0.1, 1.0, 10.0]

# Simulate quadratic loss landscape: L(w) = (w-2)^2
w_star = 2.0
n_steps = 20

fig, axes = plt.subplots(1, len(lr_values), figsize=(15, 4))

for idx, lr in enumerate(lr_values):
    w = 0.0
    ws = [w]
    for _ in range(n_steps):
        grad = 2 * (w - w_star)  # dL/dw for L=(w-2)^2
        w = w - lr * grad
        ws.append(w)

    axes[idx].plot(ws, 'o-', markersize=3, color='steelblue')
    axes[idx].axhline(w_star, color='tomato', linestyle='--', label=f'optimum={w_star}')
    axes[idx].set_title(f'lr={lr}')
    axes[idx].set_xlabel('Step')
    if idx == 0:
        axes[idx].set_ylabel('w')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim(-5, 8)

plt.suptitle('Effect of Learning Rate on L=(w-2)^2', fontsize=12)
plt.tight_layout()
plt.savefig('lr_effect.png', dpi=80, bbox_inches='tight')
plt.show()

print('lr=0.001: converges but very slowly')
print('lr=0.01:  converges nicely')
print('lr=0.1:   converges, mild oscillations')
print('lr=1.0:   may oscillate around minimum')
print('lr=10.0:  diverges (overshoots wildly)')

In [ ]:
print('='*60)
print('Q13: What is the receptive field in a CNN?')
print('='*60)
print()
print('The receptive field is the region of the INPUT IMAGE that')
print('influences a single neuron in a given layer.')
print()
print('Single conv layer with 3x3 kernel: RF = 3x3')
print('Two stacked 3x3 conv layers: RF = 5x5')
print('Three stacked 3x3 conv layers: RF = 7x7')
print()
print('RF CALCULATION:')
print('  RF_0 = 1  (pixel = sees itself)')
print('  RF_l = RF_{l-1} + (kernel_size - 1) * product_of_strides')
print()

def compute_rf(layers):
    """layers: list of (kernel_size, stride) tuples"""
    rf = 1
    stride_product = 1
    for k, s in layers:
        rf = rf + (k - 1) * stride_product
        stride_product *= s
    return rf

configs = [
    [('3x3', (3,1)), ('3x3', (3,1)), ('3x3', (3,1))],
    [('5x5', (5,1))],
    [('7x7', (7,1))]
]
print('Equivalent receptive fields:')
print(f'  3x conv3x3: RF = {compute_rf([(3,1),(3,1),(3,1)])} pixels')
print(f'  1x conv5x5: RF = {compute_rf([(5,1)])} pixels')
print(f'  1x conv7x7: RF = {compute_rf([(7,1)])} pixels')
print()
print('KEY INSIGHT: 3x conv(3x3) covers same RF as 1x conv(7x7) but with')
print('  fewer parameters: 3*(3*3*C^2) = 27C^2 vs 7*7*C^2 = 49C^2')
print('  AND more non-linearities -> richer feature learning')
print('  This is why VGG uses stacked 3x3 convolutions!')

In [ ]:
print('='*60)
print('Q14: What is backpropagation through time (BPTT)?')
print('='*60)
print()
print('BPTT is used for RNNs where the same weights are applied at each time step.')
print()
print('RNN forward pass:')
print('  h_t = tanh(W_hh * h_{t-1} + W_xh * x_t + b)')
print('  y_t = W_hy * h_t')
print()
print('BPTT unrolls the RNN through time and applies chain rule:')
print('  dL/dW = sum_{t=1}^{T} dL_t/dW')
print()
print('DIAGRAM (unrolled 3 steps):')
print('  x1   x2   x3')
print('  |    |    |')
print('  h0->h1->h2->h3')
print('       |    |    |')
print('       y1   y2   y3')
print()
print('PROBLEM: Gradients must flow back through T time steps.')
print('  With tanh: gradient * W_hh^T at each step -> vanish/explode for long T')
print()
print('SOLUTION: LSTMs and GRUs use gate mechanisms to control gradient flow.')
print('  LSTM has forget gate that can preserve gradient for long sequences.')
print()
print('Truncated BPTT: only backpropagate for K steps (not full sequence)')
print('  Trades off accuracy for speed (common in language models)')

In [ ]:
print('='*60)
print('Q15: Bias-Variance tradeoff in deep learning')
print('='*60)
print()
print('UNDERFITTING (high bias):')
print('  - Model too simple (too few layers/neurons)')
print('  - High training loss AND high validation loss')
print('  - Solution: more capacity, longer training')
print()
print('OVERFITTING (high variance):')
print('  - Model too complex, memorizes training data')
print('  - Low training loss, HIGH validation loss')
print('  - Solution: regularization (dropout, L2), more data, early stopping')
print()

# Generate bias-variance tradeoff curve
np.random.seed(42)
noise = np.random.randn(20) * 0.3
x_data = np.linspace(0, 4, 20)
y_data = np.sin(x_data) + noise
x_smooth = np.linspace(0, 4, 200)

# True function
y_true = np.sin(x_smooth)

# Underfit: degree 1 polynomial
coeffs1 = np.polyfit(x_data, y_data, 1)
y_underfit = np.polyval(coeffs1, x_smooth)

# Good fit: degree 3
coeffs3 = np.polyfit(x_data, y_data, 3)
y_goodfit = np.polyval(coeffs3, x_smooth)

# Overfit: degree 15
coeffs15 = np.polyfit(x_data, y_data, 15)
y_overfit = np.polyval(coeffs15, x_smooth)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
titles = ['Underfit (degree=1)', 'Good Fit (degree=3)', 'Overfit (degree=15)']
fits = [y_underfit, y_goodfit, y_overfit]

for ax, title, fit in zip(axes, titles, fits):
    ax.scatter(x_data, y_data, alpha=0.7, color='gray', s=30, label='data')
    ax.plot(x_smooth, y_true, 'k--', alpha=0.5, label='true')
    ax.plot(x_smooth, np.clip(fit, -3, 3), 'tomato', linewidth=2, label='model')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_ylim(-2.5, 2.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bias_variance.png', dpi=80, bbox_inches='tight')
plt.show()

print('Modern finding: deep networks in the double descent regime')
print('can overparameterize yet still generalize well (benign overfitting)')

---
## Summary Cheat Sheet

### PyTorch Must-Knows for Interviews

```
Training loop checklist:
  [ ] optimizer.zero_grad()  before each batch
  [ ] model.train()          at start of training epoch
  [ ] loss.backward()        after forward pass
  [ ] clip_grad_norm_()      if RNN or deep net
  [ ] optimizer.step()       after backward
  [ ] model.eval()           before validation
  [ ] torch.no_grad()        context during validation
```

### Key Architecture Decisions

| Problem | Architecture | Loss | Output Activation |
|---------|-------------|------|------------------|
| Binary classification | MLP | BCEWithLogitsLoss | Sigmoid |
| Multi-class | MLP/CNN | CrossEntropyLoss | None (raw logits) |
| Regression | MLP | MSELoss | None |
| Image classification | CNN | CrossEntropyLoss | None |
| Transfer learning | Pretrained + custom head | CrossEntropyLoss | None |

### Common Interview Traps

1. **Forget `optimizer.zero_grad()`** -> gradients accumulate, wrong updates
2. **Apply softmax before CrossEntropyLoss** -> double softmax, bad loss
3. **Forget `model.eval()`** -> dropout active during inference, nondeterministic
4. **Forget `.to(device)`** -> device mismatch error
5. **Use `.numpy()` on CUDA tensor** -> must call `.cpu().detach().numpy()`
6. **Modify tensor in-place while computing gradients** -> graph corruption

In [ ]:
# Final: demonstrate the common trap #5 and correct fix
if TORCH_AVAILABLE:
    t = torch.randn(5, requires_grad=True)

    # Wrong: calling .numpy() on tensor with grad
    try:
        arr = t.numpy()
        print('Direct .numpy() worked (CPU tensor, but has grad - may warn)')
    except RuntimeError as e:
        print(f'Error with direct .numpy(): {e}')

    # Correct: detach first, then convert
    arr = t.detach().numpy()
    print(f'Correct: t.detach().numpy() -> {arr[:3]}')

    # On CUDA: must also .cpu() first
    if torch.cuda.is_available():
        t_gpu = t.cuda()
        arr = t_gpu.cpu().detach().numpy()  # cpu() -> detach() -> numpy()
        print(f'GPU tensor: .cpu().detach().numpy() -> {arr[:3]}')
    else:
        print('CUDA not available; pattern: tensor.cpu().detach().numpy()')
else:
    print('PyTorch not available')
    print('Remember: tensor.cpu().detach().numpy() for safe conversion')

print()
print('Notebook complete! All 5 sections covered:')
print('  1. PyTorch Fundamentals')
print('  2. MLP from Scratch')
print('  3. CNNs + Transfer Learning')
print('  4. Training Tricks')
print('  5. 15 Interview Q&A')